# Lab 01 · Provision your AI Gateway

Create the shared AI Gateway that every later lab calls: an `aigateways` resource with a system-assigned managed
identity, plus the connector gateway that the portal creates alongside it. This notebook is the Python track of the
[lab guide](README.md): it runs the same Azure Resource Manager calls as the CLI track, end to end.

**Register providers → Create connector gateway → Create gateway → Read runtime URL → Call it**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| AI Gateway (`Microsoft.ApiManagement/aigateways`) | Runs the data plane at `https://<name>.<region>-01.ai.gateway.azure.com`. Models, tools, keys, and policies hang off it. |
| Connector gateway (`Microsoft.Web/connectorGateways`) | Companion resource with the same name. The portal creates it first, so this notebook does too. |
| System-assigned managed identity | The gateway's own Microsoft Entra identity. Lab 02 grants it access to Foundry so the gateway calls models without keys. |
| AI Gateway portal ([ai.gateway.azure.com](https://ai.gateway.azure.com)) | Where you manage the gateway after it exists. |

![AI Gateway architecture: apps and agents call one gateway that governs models and tools](../assets/diagrams/architecture.svg)

## Prerequisites

- An Azure subscription where you can create resources (Contributor on the resource group) and assign roles
  (Owner, User Access Administrator, or Role Based Access Control Administrator).
- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`. The notebook
  reuses your CLI sign-in through `AzureCliCredential`.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- Permission to register the `Microsoft.ApiManagement` and `Microsoft.Web` resource providers, or have them
  registered already.

This is the first notebook in the series. Notebooks 02, 04, 06, and 11 reuse the gateway it creates, so keep it until
you finish them.

Run the cells in order, or click **Run All**. Each step is idempotent: it reuses what already exists.

In [ ]:
%pip install -q --upgrade azure-identity requests

### 0. Initialize variables and clients

Set the region, resource group, and gateway name. The defaults match every notebook in the series, so later
notebooks find this gateway without edits. Leave `gateway_name` empty to derive a name that's unique to your
subscription, or set the `AIGW_LOCATION`, `AIGW_RESOURCE_GROUP`, and `AIGW_GATEWAY_NAME` environment variables.

The `arm` helper calls Azure Resource Manager with a token from your `az login` session, and `wait_until_ready`
polls `provisioningState` for long-running creates.

In [ ]:
import hashlib, json, os, re, subprocess, sys, time
from pathlib import Path

import requests
from azure.identity import AzureCliCredential

location       = os.environ.get("AIGW_LOCATION", "eastus2")                  # change to swedencentral if you prefer
resource_group = os.environ.get("AIGW_RESOURCE_GROUP", "rg-ai-gateway-labs") # change to use another resource group
gateway_name   = os.environ.get("AIGW_GATEWAY_NAME", "")                     # change to pick a name (empty = derived)

lab_name   = Path(globals().get("__vsc_ipynb_file__", "01-provision-gateway.ipynb")).stem
lab_labels = {"lab": lab_name, "workload": "ai-gateway-labs"}   # tags for clean-up

result = subprocess.run("az account show -o json", shell=True, capture_output=True, text=True)
if result.returncode != 0:
    sys.exit("❌ Azure CLI not installed or not signed in. Run `az login` first.")
account         = json.loads(result.stdout)
subscription_id = account["id"]
tenant_id       = account["tenantId"]
user_name       = account["user"]["name"]

suffix       = hashlib.sha1(subscription_id.encode()).hexdigest()   # stable per subscription
gateway_name = gateway_name or f"aigw-labs-{suffix[:6]}"
assert location in ("eastus2", "swedencentral"), "During the preview, AI Gateway runs in eastus2 or swedencentral."
assert re.fullmatch(r"[A-Za-z]([A-Za-z0-9-]{0,22}[A-Za-z0-9])?", gateway_name), "Use 1-24 letters, digits, or hyphens."

ARM        = "https://management.azure.com"
ARM_API    = "2025-09-01-preview"   # Microsoft.ApiManagement/aigateways
GATEWAY_ID = (f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
              f"/providers/Microsoft.ApiManagement/aigateways/{gateway_name}")

credential = AzureCliCredential()   # every client below reuses your az login
_arm_token = None

def arm(method, path, body=None, api_version=ARM_API, params=None):
    """Call Azure Resource Manager and return the JSON body (None when a GET or DELETE finds nothing)."""
    global _arm_token
    if _arm_token is None or _arm_token.expires_on - time.time() < 300:
        _arm_token = credential.get_token("https://management.azure.com/.default")
    response = requests.request(method, ARM + path, json=body, timeout=120,
                                params={"api-version": api_version, **(params or {})},
                                headers={"Authorization": f"Bearer {_arm_token.token}"})
    if response.status_code == 404 and method in ("GET", "DELETE"):
        return None
    if not response.ok:
        raise RuntimeError(f"{method} {path} -> {response.status_code}: {response.text[:1000]}")
    return response.json() if response.content else {}

def wait_until_ready(path, api_version=ARM_API, label="Resource", timeout_minutes=30):
    """Poll properties.provisioningState until it reports Succeeded."""
    start = time.time()
    while True:
        resource = arm("GET", path, api_version=api_version)
        state = "NotFound" if resource is None else resource.get("properties", {}).get("provisioningState", "Succeeded")
        if state == "Succeeded":
            return resource
        if state in ("Failed", "Canceled"):
            raise RuntimeError(f"{label} provisioning {state}: {json.dumps(resource.get('properties'))[:1000]}")
        if time.time() - start > timeout_minutes * 60:
            raise TimeoutError(f"{label} is still {state} after {timeout_minutes} minutes.")
        print(f"⏳ {label}: {state} ({int(time.time() - start)} s)")
        time.sleep(20)

print(f"🧪 Lab:            {lab_name}")
print(f"👤 User:           {user_name}")
print(f"🏛️ Tenant:         {tenant_id}")
print(f"🔑 Subscription:   {account['name']} ({subscription_id})")
print(f"🌍 Region:         {location}")
print(f"📁 Resource group: {resource_group}")
print(f"🚪 Gateway:        {gateway_name}")

### 1. Register the resource providers

AI Gateway uses two resource providers: `Microsoft.ApiManagement` for the gateway and `Microsoft.Web` for the
connector gateway. Registration is a one-time step per subscription and takes a minute or two.

In [ ]:
PROVIDERS_API = "2021-04-01"

for namespace in ("Microsoft.ApiManagement", "Microsoft.Web"):
    path  = f"/subscriptions/{subscription_id}/providers/{namespace}"
    state = arm("GET", path, api_version=PROVIDERS_API)["registrationState"]
    if state == "Registered":
        print(f"♻️ {namespace} is already registered")
        continue
    print(f"📦 Registering {namespace} ({state})")
    arm("POST", f"{path}/register", api_version=PROVIDERS_API)
    for _ in range(60):
        state = arm("GET", path, api_version=PROVIDERS_API)["registrationState"]
        if state == "Registered":
            break
        print(f"⏳ {namespace}: {state}")
        time.sleep(10)
    assert state == "Registered", f"{namespace} is {state}"
    print(f"✅ {namespace} registered")

### 2. Create the resource group

The notebook reuses the resource group if it exists. It tags the group only when it creates it, so clean-up can tell
lab resources apart.

In [ ]:
RG_PATH = f"/subscriptions/{subscription_id}/resourcegroups/{resource_group}"

group = arm("GET", RG_PATH, api_version="2021-04-01")
if group:
    print(f"♻️ Resource group {resource_group} exists in {group['location']}")
else:
    group = arm("PUT", RG_PATH, {"location": location, "tags": lab_labels}, api_version="2021-04-01")
    print(f"✅ Created resource group {resource_group} in {location}")

### 3. Create the connector gateway

The portal creates a `Microsoft.Web/connectorGateways` resource with the same name before it creates the AI gateway.
The CLI and Bicep tracks of the lab do the same, and so does this cell. The body is just a location, tags, and empty
properties.

In [ ]:
CONNECTOR_API = "2026-05-01-preview"
CONNECTOR_ID  = (f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
                 f"/providers/Microsoft.Web/connectorGateways/{gateway_name}")

if arm("GET", CONNECTOR_ID, api_version=CONNECTOR_API):
    print(f"♻️ Connector gateway {gateway_name} exists")
else:
    print(f"📦 Creating connector gateway {gateway_name}")
    arm("PUT", CONNECTOR_ID, {"location": location, "tags": lab_labels, "properties": {}}, api_version=CONNECTOR_API)

connector = wait_until_ready(CONNECTOR_ID, CONNECTOR_API, "Connector gateway")
print(f"✅ Connector gateway ready: {connector['name']}")

### 4. Create the AI gateway

The gateway is a `Microsoft.ApiManagement/aigateways` resource with a system-assigned managed identity and public
network access. The body mirrors `infra/main.bicep`. Creation takes a few minutes; the cell polls until the gateway
reports `Succeeded`.

> You can set `properties.backend.subnet.id` for outbound virtual network integration only at creation time.
> Lab 18 covers private networking.

In [ ]:
gateway_body = {
    "location": location,
    "tags": lab_labels,
    "identity": {"type": "SystemAssigned"},
    "properties": {"publicNetworkAccess": "Enabled"},
}

gateway = arm("GET", GATEWAY_ID)
if gateway:
    print(f"♻️ Gateway {gateway_name} exists ({gateway['properties'].get('provisioningState')})")
else:
    print(f"📦 Creating gateway {gateway_name} in {location}. This takes a few minutes.")
    arm("PUT", GATEWAY_ID, gateway_body)

gateway = wait_until_ready(GATEWAY_ID, label="Gateway")
print(f"✅ Gateway {gateway_name} is ready")

### 5. Read the runtime URL and managed identity

Always read the host name from `properties.frontend.defaultHostname` rather than building it. Apps call models under
`/default/models/openai/v1/` and tools under `/default/toolservers/<name>/mcp`, where `default` is the workspace.
Note the managed identity's principal ID: lab 02 grants it the Foundry User role.

In [ ]:
host         = gateway["properties"]["frontend"]["defaultHostname"]
gateway_url  = f"https://{host}"
principal_id = gateway["identity"]["principalId"]

print(f"🌐 Runtime URL:      {gateway_url}")
print(f"🤖 Models endpoint:  {gateway_url}/default/models/openai/v1/")
print(f"🧰 Tools endpoint:   {gateway_url}/default/toolservers/<name>/mcp")
print(f"🪪 Principal ID:     {principal_id}")
print(f"🏷️ Tags:             {gateway.get('tags')}")

assert re.fullmatch(rf"{re.escape(gateway_name.lower())}\.[a-z0-9]+-\d+\.ai\.gateway\.azure\.com", host.lower()), host
assert gateway["identity"]["type"].startswith("SystemAssigned")

### 6. Call the runtime URL

Send a request without a key. Any HTTP status below 500, such as `400`, `401`, or `404`, proves that DNS, TLS, and the
gateway are up. You add a model and a key in the next notebook. A brand-new host name can take a minute to resolve, so the
cell retries connection errors.

In [ ]:
url = f"{gateway_url}/default/models/openai/v1/chat/completions"
for attempt in range(20):
    try:
        response = requests.get(url, timeout=30)
        break
    except requests.ConnectionError:
        print(f"⏳ Waiting for {host} to resolve ({attempt + 1}/20)")
        time.sleep(15)
else:
    raise TimeoutError(f"{host} didn't answer. Check the gateway in the portal.")

print(f"📨 GET {url} -> HTTP {response.status_code}")
assert response.status_code < 500, response.text[:500]
print("✅ The runtime endpoint answers over HTTPS")

### 7. Explore the portal

Open the [AI Gateway portal](https://ai.gateway.azure.com) and select your gateway in the switcher in the top bar.

- **Home** shows the setup guide for the new gateway.
- **Managed identities** › **Configure identities** shows the system-assigned identity. Its principal ID matches the
  value printed in step 5.
- **Models**, **Tools**, **Policies**, and **Keys** are empty for now. Notebooks 02, 04, and 06 fill them in.

The [lab guide](README.md#tour-the-portal) has a table of every portal area and the lab that covers it.

### 8. Clean up resources

Keep the gateway: notebooks 02, 04, 06, and 11 reuse it. When you finish the series, uncomment the lines below to
delete the gateway, then the connector gateway. Gateway deletion takes about 10 minutes. Delete the resource group
only if this notebook created it and nothing else lives in it.

In [ ]:
# Later notebooks reuse the gateway. Uncomment to delete it when you're done with the series.
# arm("DELETE", GATEWAY_ID)
# while arm("GET", GATEWAY_ID):
#     print("⏳ Deleting gateway...")
#     time.sleep(30)
# print("🗑️ Gateway deleted")
# arm("DELETE", CONNECTOR_ID, api_version=CONNECTOR_API)
# print("🗑️ Connector gateway deleted")
# arm("DELETE", RG_PATH, api_version="2021-04-01")   # only if this notebook created the resource group
if arm("GET", GATEWAY_ID):
    print(f"ℹ️ Kept gateway {gateway_name} and connector gateway for the next notebooks")

## Troubleshooting

- **`The subscription is not registered to use namespace`**: rerun step 1, or run
  `az provider register -n Microsoft.ApiManagement` and `az provider register -n Microsoft.Web`.
- **`LocationNotAvailableForResourceType`**: during the preview, use `eastus2` or `swedencentral`.
- **`AuthorizationFailed`**: you need Contributor on the resource group. Check with
  `az role assignment list --assignee <your-upn> -g <resource-group> -o table`.
- **Name rejected or already taken**: the name becomes the first label of the host name. Use up to 24 letters, digits,
  and hyphens, and start with a letter. Set `AIGW_GATEWAY_NAME` to try another name.
- **`ConnectionError` or `Could not resolve host`** in step 6: the host name is new. Wait a minute and rerun the cell.
- **`az resource show` hangs** when you mix this notebook with the CLI track: call ARM directly, as the `arm` helper
  does, for example `az rest --method get --url "https://management.azure.com$GATEWAY_ID?api-version=2025-09-01-preview"`.

## Next steps

- Continue with [Publish a Foundry model and make your first call](../02-first-model-call/README.md) and its notebook [02-first-model-call.ipynb](../02-first-model-call/02-first-model-call.ipynb).
- Learn the concepts in [How AI Gateway works](../00-how-ai-gateway-works/README.md).
- Docs: [AI Gateway CLI quickstart](https://ai.gateway.azure.com/docs/quickstarts/cli).
- Read this lab on the labs site: [https://azure.github.io/ai-gateway/labs/01-provision-gateway/](https://azure.github.io/ai-gateway/labs/01-provision-gateway/).